# Smoke and Non-Smoke Day Analysis

This notebook filters air quality data (`UT_Reformated.csv`) into smoke and non-smoke days based on satellite smoke data and Salt Lake City geometry.

In [ ]:
import geopandas as gpd
import pandas as pd
from datetime import date, timedelta, datetime
import os
import warnings
warnings.filterwarnings('ignore')

## Define Helper Functions

In [ ]:
def check_fire(smoke_gdf, county_gdf, county_name='Salt Lake City'):
    """
    Checks if any smoke geometry intersects with the specified county geometry.
    Adapted from SmokeYears.py
    """
    # Filter for the specific county
    county_geom_df = county_gdf[county_gdf['NAME'] == county_name]
    
    if county_geom_df.empty:
        print(f"Warning: {county_name} not found in counties file.")
        return False
        
    county_shape = county_geom_df.geometry.iloc[0]
    
    # Check intersection with any smoke cloud
    # Using vectorized operations for efficiency
    return smoke_gdf.intersects(county_shape).any()

def get_smoke_file_path(target_date, base_path):
    """
    Constructs the path to the smoke shapefile for a given date.
    """
    date_str = f"{target_date.year}{target_date.month:02}{target_date.day:02}"
    filename = f"hms_smoke{date_str}"
    # Path structure based on user folder: Smoke_Satelite_Data/hms_smokeYYYYMMDD/hms_smokeYYYYMMDD.shp
    full_path = os.path.join(base_path, 'Smoke_Satelite_Data', filename, f"{filename}.shp")
    return full_path

## Load Data

In [ ]:
# Load Air Quality Data
data_file = '../../data/utah/filtered_by_clearing_index/Weekend_Split/SR_clearing_nonmax_summer_weekend.csv'
print(f"Loading data from {data_file}...")
df = pd.read_csv(data_file)
df['dt'] = pd.to_datetime(df['dt'])

# Load Counties Shapefile
counties_file = 'Smoke Data/Counties Shape/tl_2016_49_cousub.shp'
print(f"Loading counties from {counties_file}...")
counties = gpd.read_file(counties_file)

# Base directory for smoke data (relative to this notebook)
smoke_base_dir = 'Smoke Data'

## Process Dates

In [ ]:
smoke_dates = []
non_smoke_dates = []

unique_dates = sorted(df['dt'].dt.date.unique())
total_days = len(unique_dates)
print(f"Checking {total_days} unique days for smoke...")

for i, current_date in enumerate(unique_dates):
    if i % 50 == 0:
        print(f"Processed {i}/{total_days} days...")
        
    smoke_path = get_smoke_file_path(current_date, smoke_base_dir)
    
    is_smoke_day = False
    
    if os.path.exists(smoke_path):
        try:
            smoke_limit_gdf = gpd.read_file(smoke_path)
            if check_fire(smoke_limit_gdf, counties):
                is_smoke_day = True
        except Exception as e:
            print(f"Error reading file for {current_date}: {e}")
    else:
        # File missing usually means no smoke data, treating as non-smoke or could skip
        pass
        
    if is_smoke_day:
        smoke_dates.append(current_date)
    else:
        non_smoke_dates.append(current_date)

print("Done.")
print(f"Detected {len(smoke_dates)} smoke days.")
print(f"Detected {len(non_smoke_dates)} non-smoke days.")

## Filter and Validate

In [ ]:
# Create filtered DataFrames
smoke_dates_set = set(smoke_dates)

smoke_df = df[df['dt'].dt.date.isin(smoke_dates_set)]
non_smoke_df = df[~df['dt'].dt.date.isin(smoke_dates_set)]

# Validation
print(f"Original Row Count: {len(df)}")
print(f"Smoke Rows: {len(smoke_df)}")
print(f"Non-Smoke Rows: {len(non_smoke_df)}")

assert len(smoke_df) + len(non_smoke_df) == len(df), "Count mismatch! Rows lost or duplicated."
print("Validation Successful: Row counts match.")

# Show samples
print("\nSmoke Data Sample:")
print(smoke_df.head())

print("\nNon-Smoke Data Sample:")
print(non_smoke_df.head())

In [ ]:
# Save Results
smoke_df.to_csv('../../data/utah/filtered_by_clearing_index/Smoke_Split/Smoke/Weekend/summer.csv', index=False)
non_smoke_df.to_csv('../../data/utah/filtered_by_clearing_index/Smoke_Split/No Smoke/Weekend/summer.csv', index=False)
print("Saved 'Smoke_Days_Filtered.csv' and 'Non_Smoke_Days_Filtered.csv' to 'Smoke Data' folder.")